## 1. BigQuery 연결

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"
TABLE_ID = f"{PROJECT_ID}.{DATA_SET}.hackle_properties"

client = bigquery.Client(project=PROJECT_ID)

print(TABLE_ID)

sns-analysis-prj.sns_analysis.hackle_properties


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## 2. 삭제 대상 확인

회원 단위 분석을 위해 `user_id`가 비어 있거나 숫자로만 구성되지 않은 행을 제거합니다.  
실제 삭제 전에 각 유형의 행 수를 확인합니다.

In [3]:
sql = f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNTIF(
            user_id IS NULL
        ) AS null_user_rows,

        COUNTIF(
            user_id IS NOT NULL
            AND TRIM(user_id) = ''
        ) AS blank_user_rows,

        COUNTIF(
            TRIM(user_id) != ''
            AND NOT REGEXP_CONTAINS(TRIM(user_id), r'^[0-9]+$')
        ) AS non_numeric_user_rows,

        COUNTIF(
            user_id IS NULL
            OR TRIM(user_id) = ''
            OR NOT REGEXP_CONTAINS(TRIM(user_id), r'^[0-9]+$')
        ) AS delete_rows

    FROM `{TABLE_ID}`
"""

check_before = client.query(sql).to_dataframe()
display(check_before)

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total_rows,null_user_rows,blank_user_rows,non_numeric_user_rows,delete_rows
0,525350,0,82255,109004,191259


## 3. 결측·비숫자 user_id 제거

아래 DML 셀을 실행하면 BigQuery의 `hackle_properties` 원본 테이블에서 해당 행이 실제로 삭제됩니다.

In [4]:
sql = f"""
    DELETE FROM `{TABLE_ID}`
    WHERE user_id IS NULL
       OR TRIM(user_id) = ''
       OR NOT REGEXP_CONTAINS(TRIM(user_id), r'^[0-9]+$')
"""

query_job = client.query(sql)
query_job.result()

print(
    f"삭제 완료! 삭제된 행 수: "
    f"{query_job.num_dml_affected_rows:,}건"
)

삭제 완료! 삭제된 행 수: 191,259건


## 4. user_id 전처리 결과 확인

삭제 후 남은 행 수와 비정상 `user_id` 존재 여부를 확인합니다.

In [5]:
sql = f"""
    SELECT
        COUNT(*) AS remaining_rows,

        COUNT(DISTINCT TRIM(user_id)) AS remaining_users,

        COUNTIF(
            user_id IS NULL
            OR TRIM(user_id) = ''
            OR NOT REGEXP_CONTAINS(TRIM(user_id), r'^[0-9]+$')
        ) AS invalid_user_rows

    FROM `{TABLE_ID}`
"""

check_after = client.query(sql).to_dataframe()
display(check_after)

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,remaining_rows,remaining_users,invalid_user_rows
0,334091,230853,0


## 5. 여러 회원이 연결된 기기 확인

In [6]:
sql = f"""
SELECT
    COUNT(*) AS conflicting_device_count,
    SUM(row_count) AS affected_rows
FROM (
    SELECT
        device_id,
        COUNT(DISTINCT user_id) AS linked_user_count,
        COUNT(*) AS row_count
    FROM `{TABLE_ID}`
    WHERE device_id IS NOT NULL
      AND TRIM(device_id) != ''
    GROUP BY device_id
    HAVING COUNT(DISTINCT user_id) > 1
)
"""

client.query(sql).to_dataframe()

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,conflicting_device_count,affected_rows
0,68,195


## 6. 다중 회원 기기 데이터 제거

In [7]:
sql = f"""
DELETE FROM `{TABLE_ID}`
WHERE device_id IN (
    SELECT device_id
    FROM `{TABLE_ID}`
    WHERE device_id IS NOT NULL
      AND TRIM(device_id) != ''
    GROUP BY device_id
    HAVING COUNT(DISTINCT user_id) > 1
)
"""

query_job = client.query(sql)
query_job.result()

print("삭제된 행:", query_job.num_dml_affected_rows)

삭제된 행: 195


## 7. 최종 전처리 결과 확인

### 최종 행 수·비정상 ID 확인

In [10]:
sql = f"""
SELECT
    COUNT(*) AS remaining_rows,
    COUNTIF(
        user_id IS NULL
        OR TRIM(user_id) = ''
        OR NOT REGEXP_CONTAINS(TRIM(user_id), r'^[0-9]+$')
    ) AS invalid_user_rows
FROM `{TABLE_ID}`
"""

client.query(sql).to_dataframe()

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,remaining_rows,invalid_user_rows
0,333896,0


### 다중 회원 기기 확인

In [11]:
sql = f"""
SELECT COUNT(*) AS conflicting_device_count
FROM (
    SELECT device_id
    FROM `{TABLE_ID}`
    WHERE device_id IS NOT NULL
      AND TRIM(device_id) != ''
    GROUP BY device_id
    HAVING COUNT(DISTINCT user_id) > 1
)
"""

client.query(sql).to_dataframe()

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,conflicting_device_count
0,0


## 전처리 결과

### 1. 결측·비숫자 `user_id` 제거

* 전체 525,350건 중 `user_id`가 빈 문자열인 82,255건을 제거했습니다.
* 숫자로만 구성되지 않은 `user_id` 109,004건은 Hackle의 익명 또는 이벤트 추적용 식별자로 추정되어 회원 단위 분석에서 제외했습니다.
* 총 191,259건을 제거한 결과 334,091건이 남았습니다.

### 2. 다중 회원 기기 데이터 제거

* 숫자형 `user_id`만 남긴 후 기기별 회원 연결 관계를 다시 확인했습니다.
* 하나의 `device_id`에 여러 숫자형 회원 ID가 연결된 기기는 68개였으며, 관련 데이터는 195건이었습니다.
* 해당 데이터는 다중 계정 사용 또는 조작 가능성이 있다고 판단하여 분석 대상에서 제외했습니다.

### 3. 최종 결과

* 최종 데이터는 333,896건입니다.
* 결측·비숫자 `user_id`는 0건입니다.
* 여러 숫자형 회원 ID가 연결된 `device_id`는 0개입니다.
* 전처리는 BigQuery의 `DELETE` DML을 사용하여 적용했습니다.